# Preprocessing and Modeling

This notebook completes the machine-learning stage of the slowdown-prediction pipeline. It reads the feature datasets produced by `10_feature_engineering.ipynb`, builds preprocessing and classification pipelines, compares four candidate models on the validation split, selects the best performer, evaluates it once on the held-out test split, and saves the trained artifact.

**Scope of this notebook**
- Inputs: `train_features.csv`, `val_features.csv`, `test_features.csv` only.
- Outputs: `models/best_slowdown_model.joblib`, `reports/model_comparison.csv`.

**Leakage rules enforced here**
- Every preprocessing statistic is learned from train only.
- Validation and test are transformed with fitted preprocessors — never refit.
- Identifier and timestamp columns are excluded from model inputs.
- Model selection uses validation only; test is evaluated once after selection.


### 1. Import libraries, define paths, and fix reproducibility settings

**What the code does:** Imports modeling and visualisation libraries, resolves repository-relative paths to the three feature CSVs, and sets `RANDOM_STATE = 42` for every stochastic step.

**Why it is needed:** A single configuration block keeps the workflow portable across machines and guarantees repeatable experiments. Output directories are created up front so later saves cannot fail silently.

**How to interpret the output:** The printed paths confirm which files will be read and where the model artifact and comparison report will be written.


In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
sns.set_theme(style="whitegrid", context="notebook")

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports"

INPUT_PATHS = {
    "train": DATA_DIR / "train_features.csv",
    "validation": DATA_DIR / "val_features.csv",
    "test": DATA_DIR / "test_features.csv",
}
BEST_MODEL_PATH = MODEL_DIR / "best_slowdown_model.joblib"
METRICS_REPORT_PATH = REPORT_DIR / "model_comparison.csv"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

for split_name, path in INPUT_PATHS.items():
    if not path.is_file():
        raise FileNotFoundError(f"Missing {split_name} features: {path}")
    print(f"{split_name}: {path}")


### 2. Load the three feature datasets

**What the code does:** Reads the train, validation, and test feature CSVs into memory without modifying the upstream artifacts.

**Why it is needed:** These files already contain engineered features and the binary target `slowdown_in_5min`. This notebook consumes them as the sole modeling input — it does not recreate features, relabel rows, or merge splits.

**How to interpret the output:** Row counts and class distributions describe each split before any preprocessing or training begins.


In [ ]:
train_df = pd.read_csv(INPUT_PATHS["train"])
validation_df = pd.read_csv(INPUT_PATHS["validation"])
test_df = pd.read_csv(INPUT_PATHS["test"])

datasets = {
    "train": train_df,
    "validation": validation_df,
    "test": test_df,
}

for split_name, data in datasets.items():
    positive = int(data["slowdown_in_5min"].sum())
    negative = int((data["slowdown_in_5min"] == 0).sum())
    rate = data["slowdown_in_5min"].mean() * 100
    print(
        f"{split_name:12s} rows={len(data):,}  "
        f"positives={positive:,}  negatives={negative:,}  positive_rate={rate:.2f}%"
    )


### 3. Separate features from identifiers and the target

**What the code does:** Defines the target column, lists identifier/time columns that must never enter the model, and builds the shared numeric feature list used by all splits.

**Why it is needed:** Columns such as `id`, `machine_id`, `run_id`, `segment_id`, and `timestamp` would let the model memorise rows or exploit temporal structure already controlled by the run-level temporal split. The classifier must learn only from sensor-derived values.

**How to interpret the output:** The feature count and schema checks confirm that every split exposes the same numeric inputs and that non-numeric feature columns are absent.


In [ ]:
TARGET_COLUMN = "slowdown_in_5min"
IDENTIFIER_COLUMNS = ["id", "machine_id", "run_id", "segment_id", "timestamp"]

feature_columns = [
    column
    for column in train_df.columns
    if column not in IDENTIFIER_COLUMNS + [TARGET_COLUMN]
]

for split_name in ["validation", "test"]:
    if train_df[feature_columns].columns.tolist() != datasets[split_name][feature_columns].columns.tolist():
        raise ValueError(f"{split_name} feature schema differs from train")

non_numeric = train_df[feature_columns].select_dtypes(exclude="number").columns.tolist()
if non_numeric:
    raise ValueError(f"Non-numeric feature columns detected: {non_numeric}")

X_train = train_df[feature_columns]
y_train = train_df[TARGET_COLUMN]
X_validation = validation_df[feature_columns]
y_validation = validation_df[TARGET_COLUMN]
X_test = test_df[feature_columns]
y_test = test_df[TARGET_COLUMN]

print(f"Target column: {TARGET_COLUMN}")
print(f"Excluded identifier/time columns: {IDENTIFIER_COLUMNS}")
print(f"Number of model features: {len(feature_columns)}")


### 4. Inspect missing values and class imbalance on train

**What the code does:** Summarises missing-value counts per feature on the training set and computes the negative-to-positive ratio used for gradient-boosting imbalance correction.

**Why it is needed:** Missingness must be imputed with train statistics only. Class-balance information determines whether imbalance handling is required; the values printed here drive `class_weight` and `scale_pos_weight` settings defined later.

**How to interpret the output:** Columns with the highest missing counts (often segment-start lags or intermittently absent sensors) justify median imputation. The `_missing` indicator columns remain as features because they encode sensor availability at prediction time.


In [ ]:
missing_on_train = X_train.isna().sum().sort_values(ascending=False)
missing_on_train = missing_on_train[missing_on_train.gt(0)]

print(f"Feature columns with missing values on train: {len(missing_on_train)}")
display(missing_on_train.head(15).to_frame("missing_count"))

class_balance = y_train.value_counts().sort_index().rename("count").to_frame()
class_balance["percentage"] = (class_balance["count"] / len(y_train) * 100).round(2)
display(class_balance)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight (computed on train, used by XGBoost): {scale_pos_weight:.4f}")


### 5. Build reusable preprocessing pipelines with `ColumnTransformer`

**What the code does:** Defines a factory that returns a `ColumnTransformer` wrapping median imputation, optionally followed by `StandardScaler`, applied to all numeric feature columns.

**Why each preprocessing step is performed:**
- **Median imputation (fitted on train only):** Lag and rolling features are undefined at segment starts; some sensors are intermittently absent. The median is robust to skewed resource metrics and avoids leaking validation or test distributions into the imputation values.
- **`StandardScaler` (Logistic Regression only):** Regularised linear models are sensitive to feature scale. CPU percentages and context-switch rates live on very different numeric ranges, so scaling is required for stable convergence.
- **No scaling for tree models:** Decision-tree splits are scale-invariant. Skipping scaling preserves the original feature units and avoids unnecessary computation.
- **`ColumnTransformer` inside `Pipeline`:** Guarantees that the same columns receive the same transforms on every `fit`, `predict`, and `predict_proba` call, eliminating manual transform ordering mistakes.

**How to interpret the output:** The printed transformer objects list the exact steps each model family will apply.


In [ ]:
def build_preprocessor(*, scale: bool) -> ColumnTransformer:
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        numeric_steps.append(("scaler", StandardScaler()))
    return ColumnTransformer(
        transformers=[("numeric", Pipeline(numeric_steps), feature_columns)],
        remainder="drop",
    )

print("Linear-model preprocessing:")
print(build_preprocessor(scale=True))
print("\nTree-model preprocessing:")
print(build_preprocessor(scale=False))


### 6. Define candidate classifiers inside sklearn `Pipeline` objects

**What the code does:** Wraps each classifier with the appropriate preprocessor in a single end-to-end `Pipeline`. Logistic Regression and Random Forest always enter the comparison; XGBoost and LightGBM are added only when the corresponding package is installed.

**Why it is needed:** A unified pipeline ensures preprocessing and classification are always applied together. Fitting the pipeline on train guarantees that imputation medians, scaling parameters, and model weights all derive from training data only.

**Class imbalance handling:**
- `class_weight="balanced"` for Logistic Regression, Random Forest, and LightGBM adjusts the loss by inverse class frequency observed on train.
- `scale_pos_weight` for XGBoost uses the train negative-to-positive ratio computed in the previous section.

**How to interpret the output:** The printed model list shows every candidate that will be trained and compared.


In [ ]:
model_candidates = {
    "Logistic Regression": Pipeline(
        steps=[
            ("preprocessor", build_preprocessor(scale=True)),
            (
                "classifier",
                LogisticRegression(
                    max_iter=2000,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
    "Random Forest": Pipeline(
        steps=[
            ("preprocessor", build_preprocessor(scale=False)),
            (
                "classifier",
                RandomForestClassifier(
                    n_estimators=300,
                    min_samples_leaf=2,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    ),
}

try:
    from xgboost import XGBClassifier

    model_candidates["XGBoost"] = Pipeline(
        steps=[
            ("preprocessor", build_preprocessor(scale=False)),
            (
                "classifier",
                XGBClassifier(
                    n_estimators=300,
                    max_depth=6,
                    learning_rate=0.05,
                    subsample=0.9,
                    colsample_bytree=0.9,
                    scale_pos_weight=scale_pos_weight,
                    random_state=RANDOM_STATE,
                    eval_metric="logloss",
                    n_jobs=-1,
                ),
            ),
        ]
    )
except ImportError:
    print("XGBoost is not installed — skipping this candidate.")

try:
    from lightgbm import LGBMClassifier

    model_candidates["LightGBM"] = Pipeline(
        steps=[
            ("preprocessor", build_preprocessor(scale=False)),
            (
                "classifier",
                LGBMClassifier(
                    n_estimators=300,
                    learning_rate=0.05,
                    num_leaves=31,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                    verbose=-1,
                ),
            ),
        ]
    )
except ImportError:
    print("LightGBM is not installed — skipping this candidate.")

print("Models in the comparison:", list(model_candidates.keys()))


### 7. Fit every pipeline on the training set only

**What the code does:** Calls `fit(X_train, y_train)` for each candidate pipeline.

**Why it is needed:** This is the core anti-leakage step of the modeling stage. No validation or test row may influence imputation values, scaling parameters, or learned model weights. Validation is reserved for comparison; test remains untouched until the final evaluation.

**How to interpret the output:** One confirmation line per successfully trained model.


In [ ]:
fitted_models = {}

for model_name, pipeline in model_candidates.items():
    pipeline.fit(X_train, y_train)
    fitted_models[model_name] = pipeline
    print(f"Trained: {model_name}")


### 8. Define a reusable metrics helper

**What the code does:** Implements a small function that computes Accuracy, Precision, Recall, F1-score, and ROC AUC from true labels, hard predictions, and positive-class probabilities.

**Why it is needed:** A single helper keeps metric definitions consistent across validation and test evaluations and prevents accidental metric formula drift between models.

**How to interpret the output:** No output is expected from this cell; the function is used in the following evaluation cells.


In [ ]:
def collect_metrics(y_true, y_pred, y_proba) -> dict:
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
    }


### 9. Evaluate all models on the validation split

**What the code does:** Generates validation predictions and probabilities for every fitted pipeline, computes the five classification metrics, and stores confusion matrices for plotting.

**Why it is needed:** Validation is the decision set for model selection. It follows train chronologically and contains data from every machine, making it a realistic proxy for production performance without consuming the final test set.

**How to interpret the output:** The table ranks models by the metrics computed on this run. These are the only scores used to select the best model.


In [ ]:
validation_results = []
validation_predictions = {}

for model_name, pipeline in fitted_models.items():
    y_pred = pipeline.predict(X_validation)
    y_proba = pipeline.predict_proba(X_validation)[:, 1]
    metrics = collect_metrics(y_validation, y_pred, y_proba)
    metrics["model"] = model_name
    metrics["split"] = "validation"
    validation_results.append(metrics)
    validation_predictions[model_name] = {
        "y_pred": y_pred,
        "y_proba": y_proba,
        "confusion_matrix": confusion_matrix(y_validation, y_pred),
    }

validation_comparison = (
    pd.DataFrame(validation_results)
    .set_index("model")
    .drop(columns=["split"])
    .sort_values(["roc_auc", "f1"], ascending=False)
)

print("Validation results (computed on this run):")
display(validation_comparison.round(4))


### 10. Visual comparison of validation metrics

**What the code does:** Draws bar charts for Accuracy, Precision, Recall, F1-score, and ROC AUC across all models on the validation split.

**Why it is needed:** Numeric tables summarise performance, but charts make trade-offs visible — for example, whether higher recall comes at the cost of lower precision.

**How to interpret the output:** Taller bars indicate better scores on the metric shown. All values come from the validation evaluation in the previous cell.


In [ ]:
metric_columns = ["accuracy", "precision", "recall", "f1", "roc_auc"]
plot_df = validation_comparison.reset_index().melt(
    id_vars="model",
    value_vars=metric_columns,
    var_name="metric",
    value_name="score",
)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

palette = sns.color_palette("viridis", n_colors=len(fitted_models))
for axis, metric in zip(axes, metric_columns):
    subset = plot_df.loc[plot_df["metric"].eq(metric)]
    sns.barplot(
        data=subset,
        x="model",
        y="score",
        hue="model",
        dodge=False,
        ax=axis,
        palette=palette,
        legend=False,
    )
    axis.set_title(metric.replace("_", " ").title())
    axis.set_ylim(0, 1)
    axis.set_xlabel("")
    axis.set_ylabel("score")
    axis.tick_params(axis="x", rotation=20)

axes[-1].axis("off")
fig.suptitle("Validation metric comparison", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


### 11. Validation ROC curves and confusion matrices

**What the code does:** Overlays validation ROC curves for every model and displays a grid of confusion matrices.

**Why it is needed:** ROC curves summarise ranking quality across decision thresholds. Confusion matrices reveal the operational error profile — false alarms versus missed slowdowns — that headline metrics alone can hide.

**How to interpret the output:** Curves closer to the top-left corner indicate stronger ranking performance. Off-diagonal confusion-matrix cells are the actionable error modes.


In [ ]:
fig, axis = plt.subplots(figsize=(8, 6))
for model_name, pipeline in fitted_models.items():
    RocCurveDisplay.from_estimator(
        pipeline,
        X_validation,
        y_validation,
        name=model_name,
        ax=axis,
    )
axis.set_title("Validation ROC curves")
axis.plot([0, 1], [0, 1], linestyle="--", color="grey", linewidth=1)
plt.tight_layout()
plt.show()

n_models = len(fitted_models)
n_cols = 2
n_rows = int(np.ceil(n_models / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 4 * n_rows))
axes = np.atleast_1d(axes).flatten()

for axis, (model_name, prediction_bundle) in zip(axes, validation_predictions.items()):
    ConfusionMatrixDisplay(
        prediction_bundle["confusion_matrix"],
        display_labels=["no slowdown", "slowdown"],
    ).plot(ax=axis, colorbar=False, cmap="Blues")
    axis.set_title(f"{model_name} — validation")

for axis in axes[n_models:]:
    axis.axis("off")

fig.suptitle("Validation confusion matrices", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


### 12. Extract feature importance where the model supports it

**What the code does:** Defines a helper that reads importances from tree-based models (`feature_importances_`) or absolute coefficients from Logistic Regression, then plots the top features for each candidate trained on this run.

**Why it is needed:** Feature importance supports model explainability during review meetings. It shows which sensor signals and engineered windows the model relies on, helping stakeholders assess whether the learned patterns are plausible.

**How to interpret the output:** Models without a native importance attribute are skipped with a message. For the others, higher bars indicate stronger contribution to predictions after preprocessing.


In [ ]:
def extract_importance(pipeline: Pipeline, model_name: str) -> pd.Series | None:
    classifier = pipeline.named_steps["classifier"]
    if hasattr(classifier, "feature_importances_"):
        values = classifier.feature_importances_
        label = "feature_importance"
    elif hasattr(classifier, "coef_"):
        values = np.abs(classifier.coef_).ravel()
        label = "abs_coefficient"
    else:
        print(f"{model_name}: no native feature importance available — skipped.")
        return None
    series = pd.Series(values, index=feature_columns, name=label).sort_values(ascending=False)
    print(f"{model_name}: importance extracted ({label}).")
    return series


importance_by_model = {}
for model_name, pipeline in fitted_models.items():
    importance = extract_importance(pipeline, model_name)
    if importance is not None:
        importance_by_model[model_name] = importance

if importance_by_model:
    n_importance = len(importance_by_model)
    n_cols = 2
    n_rows = int(np.ceil(n_importance / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4.5 * n_rows))
    axes = np.atleast_1d(axes).flatten()
    top_n = 15

    for axis, (model_name, importance) in zip(axes, importance_by_model.items()):
        top_features = importance.head(top_n).sort_values()
        axis.barh(top_features.index, top_features.values, color="steelblue")
        axis.set_title(f"Top {top_n} features — {model_name}")
        axis.set_xlabel(importance.name)

    for axis in axes[n_importance:]:
        axis.axis("off")

    fig.suptitle("Feature importance by model (train-fitted)", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No feature-importance plots were produced.")


### 13. Select the best model using validation results only

**What the code does:** Ranks validation results by ROC AUC (primary criterion) and F1-score (tie-breaker), then records the winning pipeline.

**Why it is needed:** Model selection must not use the test set. ROC AUC is threshold-independent and suitable for imbalanced classification; F1 resolves ties between models with similar ranking performance.

**How to interpret the output:** The printed model name is the sole candidate that will be evaluated on test and persisted to disk.


In [ ]:
selection_table = validation_comparison.sort_values(
    ["roc_auc", "f1"],
    ascending=False,
)
best_model_name = selection_table.index[0]
best_model = fitted_models[best_model_name]

print("Model selection based on validation metrics from this run:")
display(selection_table.round(4))
print(f"Selected best model: {best_model_name}")


### 14. Final evaluation on the held-out test split

**What the code does:** Applies the selected pipeline — already fitted on train — to the test features and reports the same five metrics plus a classification report.

**Why it is needed:** The test split provides an unbiased estimate of performance on the most recent data. It was not used for preprocessing, model selection, or hyperparameter tuning.

**How to interpret the output:** Test scores are reported once, after selection. Differences between validation and test performance indicate how well the chosen model generalises forward in time.


In [ ]:
y_test_pred = best_model.predict(X_test)
y_test_proba = best_model.predict_proba(X_test)[:, 1]

test_metrics = collect_metrics(y_test, y_test_pred, y_test_proba)
test_metrics["model"] = best_model_name
test_metrics["split"] = "test"

print(f"Final test evaluation — {best_model_name}")
display(pd.DataFrame([test_metrics]).round(4))

print(
    classification_report(
        y_test,
        y_test_pred,
        target_names=["no slowdown (0)", "slowdown (1)"],
        digits=4,
    )
)


### 15. Test-set ROC curve, confusion matrix, and feature importance

**What the code does:** Displays the ROC curve and confusion matrix for the selected model on test, then plots the top contributing features for that model when importance is available.

**Why it is needed:** These are the primary deliverable charts for a project review: they show production-relevant errors on unseen future data and highlight which inputs drive the final model.

**How to interpret the output:** Compare these plots with their validation counterparts to judge stability. The feature-importance chart applies to the saved model only.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

RocCurveDisplay.from_estimator(
    best_model,
    X_test,
    y_test,
    ax=axes[0],
    name=best_model_name,
)
axes[0].set_title(f"Test ROC — {best_model_name}")
axes[0].plot([0, 1], [0, 1], linestyle="--", color="grey", linewidth=1)

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_test_pred,
    display_labels=["no slowdown", "slowdown"],
    ax=axes[1],
    colorbar=False,
    cmap="Blues",
)
axes[1].set_title(f"Test confusion matrix — {best_model_name}")

plt.tight_layout()
plt.show()

if best_model_name in importance_by_model:
    best_importance = importance_by_model[best_model_name].head(20).sort_values()
    fig, axis = plt.subplots(figsize=(10, 8))
    axis.barh(best_importance.index, best_importance.values, color="darkorange")
    axis.set_title(f"Top 20 features — {best_model_name} (selected model)")
    axis.set_xlabel(importance_by_model[best_model_name].name)
    plt.tight_layout()
    plt.show()
else:
    print(f"Feature importance is not available for {best_model_name}.")


### 16. Save the model comparison report

**What the code does:** Writes a CSV containing validation metrics for every candidate and test metrics for the selected model only.

**Why it is needed:** A persistent comparison table supports project documentation and makes it easy to share results without re-running the notebook.

**How to interpret the output:** The saved path and preview confirm that validation rows cover all models and that exactly one test row exists for the selected model.


In [ ]:
comparison_report = pd.concat(
    [
        pd.DataFrame(validation_results),
        pd.DataFrame([test_metrics]),
    ],
    ignore_index=True,
)

comparison_report.to_csv(METRICS_REPORT_PATH, index=False)
print(f"Saved model comparison report to {METRICS_REPORT_PATH}")
display(comparison_report.round(4))


### 17. Save the best fitted pipeline with joblib

**What the code does:** Serialises the complete sklearn `Pipeline` (preprocessor + classifier) together with metadata describing the feature list, target, and metrics computed on this run.

**Why it is needed:** Deployment must apply the same train-fitted imputation and scaling. Saving the full pipeline — not just the classifier — guarantees consistent inference behaviour.

**How to interpret the output:** The saved path and file size confirm that the artifact is ready for downstream inference scripts or services.


In [ ]:
artifact = {
    "pipeline": best_model,
    "model_name": best_model_name,
    "target_column": TARGET_COLUMN,
    "feature_columns": feature_columns,
    "identifier_columns": IDENTIFIER_COLUMNS,
    "random_state": RANDOM_STATE,
    "selection_criteria": "validation roc_auc (primary), validation f1 (tie-breaker)",
    "validation_metrics": validation_comparison.loc[best_model_name].to_dict(),
    "test_metrics": {
        key: test_metrics[key]
        for key in ["accuracy", "precision", "recall", "f1", "roc_auc"]
    },
}

joblib.dump(artifact, BEST_MODEL_PATH)
print(f"Saved best model artifact to {BEST_MODEL_PATH}")
print(f"File size: {BEST_MODEL_PATH.stat().st_size / 1024:.1f} KB")


### 18. Final summary

**What this notebook accomplished**
1. Loaded the three feature datasets without altering upstream artifacts.
2. Excluded identifiers and timestamps; retained numeric sensor and engineered features only.
3. Fitted median imputation (and scaling for Logistic Regression) on train only.
4. Compared Logistic Regression, Random Forest, XGBoost, and LightGBM when available.
5. Selected the best model using validation metrics from this run.
6. Evaluated the selected model once on the held-out test split.
7. Saved the comparison report and the trained pipeline artifact.

**Caveats carried forward from earlier notebooks:** the provisional Rule C label definition and the relatively small validation split remain review points. Treat these results as structured baselines until the team confirms label semantics and split sizing.
